# Système RAG avec LangChain et Hugging Face

Ce notebook documente la création d'un système de génération augmentée par récupération (RAG). Ce système permet d'interroger un jeu de données spécifique en utilisant la puissance des modèles de langue (LLM) combinée à une recherche sémantique efficace.

## 1. Configuration de l'environnement

Nous installons les bibliothèques nécessaires :
* `langchain` : Pour l'orchestration globale.
* `transformers` & `torch` : Pour manipuler les modèles de deep learning.
* `sentence-transformers` : Pour la génération d'embeddings.
* `datasets` : Pour charger le dataset Dolly-15k.
* `faiss-cpu` : Pour le stockage et la recherche vectorielle.

In [29]:
# Installation forcée des modules spécifiques pour garantir la disponibilité des chaînes
!pip install -q -U langchain langchain-community langchain-huggingface langchain-text-splitters faiss-cpu datasets transformers torch

## 2. Chargement du jeu de données

Nous utilisons le dataset `databricks/databricks-dolly-15k`. Nous nous concentrons sur la colonne `context` qui contient les informations textuelles que nous voulons indexer.

In [2]:
from langchain_community.document_loaders import HuggingFaceDatasetLoader

# Configuration du dataset
dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

# Initialisation du chargeur spécifique à Hugging Face
loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)

# Chargement des données sous forme de documents LangChain
data = loader.load()

# Affichage des deux premières entrées pour vérification
print(f"Nombre total de documents chargés : {len(data)}")
print(data[:2])

/tmp/ipykernel_26305/2458679213.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import HuggingFaceDatasetLoader


README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Nombre total de documents chargés : 15011
[Document(metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}, page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia\'s domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."'), Document(metadata={'instruction': 'Which is a species of fish? Tope or Rope', 'response': 'Tope', 'category': 'classification'}, page_content='""')]


## 3. Fractionner les documents (Chunking)

Pour que le modèle puisse traiter le texte, nous devons diviser les documents en morceaux plus petits (chunks). Le `overlap` permet de garder un peu de contexte entre deux morceaux consécutifs.

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Création du splitter : on découpe par blocs de 1000 caractères avec 150 caractères de chevauchement
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

# Application du découpage sur nos documents
docs = text_splitter.split_documents(data)

print(f"Nombre de segments créés : {len(docs)}")
print(f"Aperçu du premier segment :\n{docs[0]}")

Nombre de segments créés : 18502
Aperçu du premier segment :
page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."' metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [16]:
print(f"Aperçu du deuxième segment :\n{docs[1]}")

Aperçu du deuxième segment :
page_content='""' metadata={'instruction': 'Which is a species of fish? Tope or Rope', 'response': 'Tope', 'category': 'classification'}


## 4. Intégration du texte (Embeddings)

Les embeddings transforment le texte en vecteurs numériques. Des textes ayant un sens similaire auront des vecteurs proches dans l'espace.

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

# Définition du modèle d'embedding (MiniLM est léger et performant)
modelPath = "sentence-transformers/all-MiniLM-l6-v2"
model_kwargs = {'device':'cpu'}
encode_kwargs = {'normalize_embeddings': False}

# Initialisation de l'outil d'embedding avec la nouvelle classe recommandée
embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# Test rapide de l'embedding sur une requête
text = "Ceci est un document de test."
query_result = embeddings.embed_query(text)
print(f"Dimension de l'embedding : {len(query_result)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Dimension de l'embedding : 384


## 5. Créer un entrepôt de vecteurs (Vector Store)

FAISS permet de stocker ces vecteurs et d'effectuer une recherche de similarité extrêmement rapide.

In [15]:
from langchain_community.vectorstores import FAISS

# Nous utilisons les documents 'docs' générés à l'étape 3
# Note : L'indexation peut prendre un moment selon la taille du dataset
try:
    db = FAISS.from_documents(docs, embeddings)
    print("Base de données vectorielle FAISS créée avec succès.")
except NameError:
    print("Erreur : Assurez-vous d'avoir exécuté la cellule de fractionnement (docs) et d'embeddings.")

Base de données vectorielle FAISS créée avec succès.


## 6. Préparation du modèle LLM

Nous utilisons un modèle spécialisé dans le Question-Answering : `dynamic_tinybert`.

In [138]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain_huggingface import HuggingFacePipeline

model_id = "Intel/dynamic_tinybert"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForQuestionAnswering.from_pretrained(model_id)

# Initialisation via la factory pipeline standard en forçant la tâche textuelle
# En passant les objets model et tokenizer, on contourne les erreurs de registre
qa_pipe = pipeline(
    task="question-answering",
    model=model,
    tokenizer=tokenizer,
    device=-1
)

llm = HuggingFacePipeline(pipeline=qa_pipe)

ModuleNotFoundError: No module named 'transformers.pipelines.question_answering'

## 7. Création de la chaîne de recherche (RetrievalQA)

C'est ici que nous lions le moteur de recherche (Retriever) au modèle de langue (LLM).

In [137]:
retriever = db.as_retriever(search_kwargs={"k": 3})

class RAGSystem:
    def __init__(self, llm, retriever):
        self.llm = llm
        self.retriever = retriever

    def invoke(self, inputs):
        query = inputs.get("query", "")
        docs = self.retriever.invoke(query)
        context = "\n".join([d.page_content for d in docs])

        # Passage des arguments nommés requis par la pipeline QuestionAnswering
        result = self.llm.pipeline(question=query, context=context)
        return {"result": result.get("answer", "Aucune réponse trouvée.")}

qa = RAGSystem(llm, retriever)
print("Système RAG prêt.")

Système RAG prêt.


## 8. Test du système RAG

Enfin, nous posons une question pour vérifier si le système extrait bien les informations du dataset pour répondre.

In [120]:
# Question de test
question = "What is cheesemaking?"

# Exécution de la requête
print(f"Question : {question}")
result = qa.invoke({"query": question})

# Affichage du résultat final généré
print("\nRéponse du système :")
print(result["result"])

Question : What is cheesemaking?


TypeError: DocumentQuestionAnsweringPipeline.__call__() missing 1 required positional argument: 'image'